# 04 — Explainability + Clinical Error Analysis (Phase 5)

Grad-CAM (via `pytorch-grad-cam`) on the last conv block (`model.layer4[-1]`) of the trained baseline, reviewed on 5 correct + 5 incorrect test-set predictions.

In [ ]:
import sys, json
sys.path.insert(0, "..")
import torch
from torch.utils.data import DataLoader

from src.data_utils import load_dataset_metadata, train_val_test_split
from src.train import HYGDDataset, build_model
from src.evaluate import evaluate
from src.visualize import grad_cam_overlay, review_predictions

df = load_dataset_metadata("../data/raw")
train_df, val_df, test_df = train_val_test_split(df)
test_ds = HYGDDataset(test_df)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False)

model = build_model(num_classes=2, freeze_backbone=True)
model.load_state_dict(torch.load("../results/baseline_resnet18.pt", map_location="cpu"))
model.eval()

metrics = evaluate(model, test_loader, device="cpu")
sample = review_predictions(metrics["y_true"], metrics["y_prob"], n_correct=5, n_wrong=5)
sample

## Correct predictions

![correct](../figures/08_gradcam_correct.png)

The heatmap concentrates on the optic disc / peripapillary region in most cases — clinically the right area to look at for glaucomatous cupping, which is reassuring (the model isn't just picking up on unrelated image artifacts).

## Incorrect predictions — clinical reflection

![wrong](../figures/09_gradcam_wrong.png)

_TODO: for each wrong prediction, look at where the heatmap focused vs. where a real clinician would look, and write one sentence per case on a plausible reason for the error (e.g. poor image quality/low FundusQ-Net score, atypical disc appearance, borderline cupping, heatmap missing the disc entirely). This is a judgment call that needs a human (ideally with real clinical/ophthalmology input, e.g. from a mentor), not something to auto-generate — left as the next real step._

## Limitations recap (see README.md for the full list)

- Single dataset (HYGD), single hospital (Hillel Yaffe Medical Center) — no external validation.
- Small test set: 99 images / 44 patients.
- Baseline only (frozen backbone, 8 epochs) — not tuned for maximum performance.
- **This is a student portfolio/research artifact, not a clinical device.**